# Session 3 — Multi-Agent Orchestration: Beige Book Comparison Index

**Workshop:** AI Agents Workshop for Master of Quantitative Economics  
**Session:** Multi-Agent Orchestration  
**Use case:** A low-cost *Macro Research Crew* that gathers the **latest three Federal Reserve Beige Books**, extracts district-level economic tone, and builds a **comparison index** across releases.

## Why this example works for quantitative economics students

The Beige Book is qualitative, regional, and timely. That makes it a good classroom object for agentic workflows because the problem naturally decomposes into multiple tasks:

1. **Collect** the latest releases.
2. **Parse** district sections.
3. **Extract** economic tone and uncertainty.
4. **Normalize** across districts and releases.
5. **Compare** changes over time.
6. **Validate** the pipeline.

The baseline below runs without an LLM API key. That keeps the demo cheap and reproducible. Optional LLM calls can later be added only where they add marginal value, such as evidence extraction or narrative summary.

## Releases used

As of the workshop date, the Federal Reserve Beige Book page lists the latest 2026 releases as:

- January 14, 2026
- March 4, 2026
- April 15, 2026

The code below uses those three releases by default and can be changed to pull more historical editions.

In [ ]:
# If running in a fresh environment, uncomment:
# !pip install requests beautifulsoup4 pandas numpy matplotlib lxml

import re
import math
import json
import textwrap
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
from collections import defaultdict

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 140)
plt.rcParams['figure.figsize'] = (10, 5)

## 1. Architecture: the Macro Research Crew

This is a multi-agent system even though it is implemented in ordinary Python classes. The important design feature is not the brand name of an agent framework; it is the separation of responsibilities and testable handoffs.

| Agent | Responsibility | Output |
|---|---|---|
| Planner | Define objective and workflow | Ordered task list |
| Collector | Get the latest 3 Beige Book HTML pages | Release objects |
| Parser | Extract district sections | District-section table |
| Sentiment Extractor | Count positive, negative, uncertainty language | Section-level scores |
| Index Builder | Normalize and aggregate | Comparison indices |
| QA Agent | Check coverage, missing sections, implausible values | QA report |

The teaching point: multi-agent orchestration is a **software design pattern for complex analytical work**, not merely a longer prompt.

In [ ]:
FED_DISTRICTS = [
    'Boston', 'New York', 'Philadelphia', 'Cleveland', 'Richmond', 'Atlanta',
    'Chicago', 'St. Louis', 'Minneapolis', 'Kansas City', 'Dallas', 'San Francisco'
]

# Latest three releases listed on the Fed Beige Book page as of May 6, 2026.
# The March report is the second 2026 Beige Book, so its Fed URL uses 202602.
LATEST_THREE_RELEASES = [
    {'release_date': '2026-01-14', 'label': 'Jan 14, 2026', 'url': 'https://www.federalreserve.gov/monetarypolicy/beigebook202601-summary.htm'},
    {'release_date': '2026-03-04', 'label': 'Mar 4, 2026',  'url': 'https://www.federalreserve.gov/monetarypolicy/beigebook202602-summary.htm'},
    {'release_date': '2026-04-15', 'label': 'Apr 15, 2026', 'url': 'https://www.federalreserve.gov/monetarypolicy/beigebook202604-summary.htm'},
]

@dataclass
class Release:
    release_date: str
    label: str
    url: str
    html: Optional[str] = None

@dataclass
class DistrictSection:
    release_date: str
    label: str
    district: str
    text: str

## 2. Collector Agent

The collector is intentionally simple: it downloads three official Federal Reserve HTML pages. For classroom reliability, the URLs are explicit rather than discovered by a fragile live scrape.

In [ ]:
class CollectorAgent:
    def __init__(self, releases=LATEST_THREE_RELEASES, timeout=30):
        self.releases = [Release(**r) for r in releases]
        self.timeout = timeout

    def run(self) -> List[Release]:
        downloaded = []
        for r in self.releases:
            resp = requests.get(r.url, timeout=self.timeout, headers={'User-Agent': 'Mozilla/5.0'})
            resp.raise_for_status()
            r.html = resp.text
            downloaded.append(r)
        return downloaded

## 3. Parser Agent

The parser converts each report into district-level sections. This is the handoff that matters most: if district parsing fails, every downstream index is invalid.

In [ ]:
class ParserAgent:
    def html_to_text(self, html: str) -> str:
        soup = BeautifulSoup(html, 'html.parser')
        for tag in soup(['script', 'style', 'nav', 'footer', 'noscript']):
            tag.decompose()
        text = soup.get_text('\n')
        text = re.sub(r'\r', '\n', text)
        text = re.sub(r'\n{3,}', '\n\n', text)
        text = re.sub(r'[ \t]+', ' ', text)
        return text

    def extract_sections(self, release: Release) -> List[DistrictSection]:
        text = self.html_to_text(release.html or '')
        # Start at the first district highlight to avoid publication front matter.
        heading_pattern = re.compile(r'(?m)^\s*(' + '|'.join(map(re.escape, FED_DISTRICTS)) + r')\s*$')
        matches = list(heading_pattern.finditer(text))
        sections = []
        for i, m in enumerate(matches):
            district = m.group(1)
            start = m.end()
            end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
            section_text = text[start:end].strip()
            # Summary-page district highlights are short but still usable for the classroom index.
            if len(section_text.split()) >= 20:
                sections.append(DistrictSection(release.release_date, release.label, district, section_text))
        return sections

    def run(self, releases: List[Release]) -> pd.DataFrame:
        rows = []
        for r in releases:
            for s in self.extract_sections(r):
                rows.append({
                    'release_date': s.release_date,
                    'label': s.label,
                    'district': s.district,
                    'text': s.text,
                    'tokens': len(re.findall(r"[A-Za-z]+(?:-[A-Za-z]+)?", s.text)),
                })
        return pd.DataFrame(rows)

## 4. Sentiment Extractor Agent

We begin with a transparent dictionary method. This is not the final word in NLP; it is the baseline that students can audit and improve.

### Raw district sentiment

\[
S_{d,t}=100\times\frac{P_{d,t}-N_{d,t}}{\sqrt{\text{tokens}_{d,t}}}
\]

where \(P_{d,t}\) is the count of positive economic-tone terms and \(N_{d,t}\) is the count of negative economic-tone terms for district \(d\) in release \(t\).

### Uncertainty rate

\[
U_{d,t}=1000\times\frac{\text{uncertainty terms}_{d,t}}{\text{tokens}_{d,t}}
\]

In [ ]:
class SentimentExtractorAgent:
    POSITIVE = set('''
        accelerate accelerated accelerating advance advanced advancing benefit beneficial boom boosted bright constructive eased easing elevated expand expanded expanding expansion
        favorable gain gained gains grow growing growth healthy improve improved improvement improving increase increased increasing optimistic outperform outperformed positive
        rebound rebounded recover recovered recovery resilient robust solid stable stabilized stabilization strength strengthened strengthening strong stronger tight upside
    '''.split())

    NEGATIVE = set('''
        adverse bankrupt bankruptcy contraction contracted contracting decline declined declining decrease decreased decreasing deteriorate deteriorated deterioration downside
        fall falling fell fragile headwind headwinds layoff layoffs negative recession recessionary reduce reduced reducing soft soften softened softening strain strained stress
        stressed sluggish slowdown slowed slowing weak weakened weakening weaker weakness
    '''.split())

    UNCERTAIN = set('''
        ambiguity ambiguous caution cautious concern concerned concerns conflict disruption disruptions geopolitical hesitant mixed risk risks uncertain uncertainty unclear uneven
        unpredictable volatile volatility wait-and-see tariff tariffs inflationary
    '''.split())

    TOPIC_KEYWORDS = {
        'Labor': set('labor employment hiring worker workers wage wages payroll layoffs unemployment job jobs staffing'),
        'Prices': set('price prices inflation inflationary cost costs input inputs margin margins tariff tariffs'),
        'Consumer': set('consumer consumers spending retail auto autos tourism travel restaurants leisure'),
        'Manufacturing': set('manufacturing factory factories production orders supply inventories'),
        'Real Estate': set('housing residential commercial construction real estate rent rents mortgage office'),
        'Credit/Finance': set('loan loans credit banking bank banks delinquency delinquencies deposit deposits financial'),
    }

    def tokenize(self, text: str) -> List[str]:
        return re.findall(r"[a-z]+(?:-[a-z]+)?", text.lower())

    def sentence_split(self, text: str) -> List[str]:
        return re.split(r'(?<=[.!?])\s+', text)

    def score_text(self, text: str) -> Dict[str, float]:
        toks = self.tokenize(text)
        n = max(len(toks), 1)
        pos = sum(t in self.POSITIVE for t in toks)
        neg = sum(t in self.NEGATIVE for t in toks)
        unc = sum(t in self.UNCERTAIN for t in toks)
        return {
            'positive_count': pos,
            'negative_count': neg,
            'uncertainty_count': unc,
            'token_count': n,
            'raw_sentiment': 100 * (pos - neg) / math.sqrt(n),
            'uncertainty_rate': 1000 * unc / n,
        }

    def topic_scores(self, text: str) -> List[Dict[str, float]]:
        sentences = self.sentence_split(text)
        rows = []
        for topic, kws in self.TOPIC_KEYWORDS.items():
            topic_text = ' '.join(s for s in sentences if set(self.tokenize(s)) & kws)
            if len(topic_text.split()) >= 15:
                score = self.score_text(topic_text)
                score['topic'] = topic
                rows.append(score)
        return rows

    def run(self, sections: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
        section_rows = []
        topic_rows = []
        for _, row in sections.iterrows():
            score = self.score_text(row['text'])
            section_rows.append({
                'release_date': row['release_date'],
                'label': row['label'],
                'district': row['district'],
                **score,
            })
            for ts in self.topic_scores(row['text']):
                topic_rows.append({
                    'release_date': row['release_date'],
                    'label': row['label'],
                    'district': row['district'],
                    **ts,
                })
        return pd.DataFrame(section_rows), pd.DataFrame(topic_rows)

## 5. Index Builder Agent

The comparison index is designed for a three-release classroom example.

For each district, we standardize its sentiment over the three releases:

\[
z_{d,t}=\frac{S_{d,t}-\bar S_d}{\sigma_d}
\]

Then the release-level comparison index is:

\[
\text{BB Comparison Index}_t = 100 + 10\times\frac{1}{12}\sum_d z_{d,t}
\]

Interpretation:

- **Above 100**: tone is more positive than the district-adjusted three-release average.
- **Below 100**: tone is less positive than the district-adjusted three-release average.
- **Change from prior release**: macro narrative momentum.

This avoids pretending that three observations are enough for a full historical z-score. It is a short-window comparison measure.

In [ ]:
class IndexBuilderAgent:
    def _safe_z_by_group(self, df, value_col, group_col, new_col):
        out = df.copy()
        def z(x):
            sd = x.std(ddof=0)
            if sd == 0 or np.isnan(sd):
                return x * 0
            return (x - x.mean()) / sd
        out[new_col] = out.groupby(group_col)[value_col].transform(z)
        return out

    def district_comparison_index(self, section_scores: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
        df = section_scores.copy()
        df['release_date'] = pd.to_datetime(df['release_date'])
        df = self._safe_z_by_group(df, 'raw_sentiment', 'district', 'district_time_z')
        df['district_comparison_index'] = 100 + 10 * df['district_time_z']

        release_index = (df.groupby(['release_date', 'label'], as_index=False)
                           .agg(
                               bb_comparison_index=('district_comparison_index', 'mean'),
                               mean_raw_sentiment=('raw_sentiment', 'mean'),
                               mean_uncertainty_rate=('uncertainty_rate', 'mean'),
                               districts=('district', 'nunique'),
                               tokens=('token_count', 'sum')
                           )
                           .sort_values('release_date'))
        release_index['index_change_prior'] = release_index['bb_comparison_index'].diff()
        release_index['uncertainty_change_prior'] = release_index['mean_uncertainty_rate'].diff()
        return df.sort_values(['release_date', 'district']), release_index

    def topic_comparison_index(self, topic_scores: pd.DataFrame) -> pd.DataFrame:
        if topic_scores.empty:
            return topic_scores
        df = topic_scores.copy()
        df['release_date'] = pd.to_datetime(df['release_date'])
        # Standardize within topic so each topic's scale is comparable across three releases.
        df = self._safe_z_by_group(df, 'raw_sentiment', 'topic', 'topic_time_z')
        df['topic_comparison_index'] = 100 + 10 * df['topic_time_z']
        topic_release = (df.groupby(['release_date', 'label', 'topic'], as_index=False)
                           .agg(topic_comparison_index=('topic_comparison_index', 'mean'),
                                uncertainty_rate=('uncertainty_rate', 'mean'),
                                observations=('district', 'nunique'))
                           .sort_values(['topic', 'release_date']))
        topic_release['topic_change_prior'] = topic_release.groupby('topic')['topic_comparison_index'].diff()
        return topic_release

    def run(self, section_scores: pd.DataFrame, topic_scores: pd.DataFrame) -> Dict[str, pd.DataFrame]:
        district_index, release_index = self.district_comparison_index(section_scores)
        topic_index = self.topic_comparison_index(topic_scores)
        return {
            'district_index': district_index,
            'release_index': release_index,
            'topic_index': topic_index,
        }

## 6. QA Agent

A credible agent workflow must check its own outputs. The QA agent does not “make the analysis look good”; it tells us whether the pipeline is trustworthy enough to interpret.

In [ ]:
class QAAgent:
    def run(self, sections: pd.DataFrame, section_scores: pd.DataFrame, indices: Dict[str, pd.DataFrame]) -> Dict:
        expected = len(LATEST_THREE_RELEASES) * len(FED_DISTRICTS)
        actual_sections = len(sections)
        district_counts = sections.groupby('label')['district'].nunique().to_dict() if not sections.empty else {}
        min_tokens = int(section_scores['token_count'].min()) if not section_scores.empty else 0
        max_tokens = int(section_scores['token_count'].max()) if not section_scores.empty else 0

        missing = {}
        for label in [r['label'] for r in LATEST_THREE_RELEASES]:
            present = set(sections.loc[sections['label'] == label, 'district'])
            missing[label] = sorted(set(FED_DISTRICTS) - present)

        flags = []
        if actual_sections != expected:
            flags.append(f'Expected {expected} district sections, found {actual_sections}.')
        if min_tokens < 20:
            flags.append(f'At least one district highlight has unusually few tokens: min_tokens={min_tokens}.')
        if indices['release_index']['districts'].min() < 12:
            flags.append('At least one release has fewer than 12 districts in the index.')

        return {
            'expected_sections': expected,
            'actual_sections': actual_sections,
            'districts_per_release': district_counts,
            'missing_districts': missing,
            'min_tokens': min_tokens,
            'max_tokens': max_tokens,
            'flags': flags or ['No major QA flags.'],
        }

## 7. Run the crew

Run this cell to build the comparison index from the latest three Beige Books.

In [ ]:
collector = CollectorAgent()
parser = ParserAgent()
extractor = SentimentExtractorAgent()
indexer = IndexBuilderAgent()
qa = QAAgent()

releases = collector.run()
sections = parser.run(releases)
section_scores, topic_scores = extractor.run(sections)
indices = indexer.run(section_scores, topic_scores)
qa_report = qa.run(sections, section_scores, indices)

print(json.dumps(qa_report, indent=2))

## 8. Main output: release-level comparison index

This table answers: **Did the overall Beige Book tone improve or deteriorate across the last three releases?**

In [ ]:
release_index = indices['release_index']
release_index

## 9. District-level comparison table

This table answers: **Which districts drove the aggregate change?**

In [ ]:
district_pivot = (indices['district_index']
    .pivot_table(index='district', columns='label', values='district_comparison_index')
    .reindex(FED_DISTRICTS))

district_pivot['Change: latest minus first'] = district_pivot.iloc[:, -1] - district_pivot.iloc[:, 0]
district_pivot.sort_values('Change: latest minus first')

## 10. Topic-level comparison index

This table answers: **Was the shift concentrated in prices, labor, consumers, manufacturing, real estate, or credit?**

In [ ]:
topic_index = indices['topic_index']
topic_pivot = topic_index.pivot_table(index='topic', columns='label', values='topic_comparison_index')
topic_pivot['Change: latest minus first'] = topic_pivot.iloc[:, -1] - topic_pivot.iloc[:, 0]
topic_pivot.sort_values('Change: latest minus first')

## 11. Visualize the comparison index

In [ ]:
ax = release_index.plot(
    x='label', y='bb_comparison_index', marker='o', legend=False,
    title='Beige Book Comparison Index: Latest Three Releases'
)
ax.axhline(100, linestyle='--', linewidth=1)
ax.set_xlabel('Release')
ax.set_ylabel('Index: 100 = three-release district-adjusted average')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
mat = district_pivot.drop(columns=['Change: latest minus first']).values
im = ax.imshow(mat, aspect='auto')
ax.set_xticks(range(len(district_pivot.columns) - 1))
ax.set_xticklabels(district_pivot.columns[:-1], rotation=0)
ax.set_yticks(range(len(district_pivot.index)))
ax.set_yticklabels(district_pivot.index)
ax.set_title('District Comparison Index Heatmap')
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j, i, f'{mat[i, j]:.1f}', ha='center', va='center')
fig.colorbar(im, ax=ax, label='Index')
plt.tight_layout()
plt.show()

In [ ]:
if not topic_index.empty:
    latest_label = release_index.iloc[-1]['label']
    first_label = release_index.iloc[0]['label']
    delta = topic_pivot['Change: latest minus first'].sort_values()
    ax = delta.plot(kind='barh', title=f'Topic Momentum: {latest_label} minus {first_label}')
    ax.axvline(0, linewidth=1)
    ax.set_xlabel('Index-point change')
    plt.tight_layout()
    plt.show()

## 12. Evidence extraction: show the text behind the weakest and strongest districts

This keeps the index from becoming a black box. Students should inspect excerpts before interpreting results.

In [ ]:
latest_label = release_index.iloc[-1]['label']
latest = indices['district_index'].query('label == @latest_label').sort_values('district_comparison_index')
weakest = latest.iloc[0]['district']
strongest = latest.iloc[-1]['district']

for district in [weakest, strongest]:
    txt = sections.query('label == @latest_label and district == @district')['text'].iloc[0]
    print('\n' + '='*90)
    print(f'{latest_label} — {district}')
    print('='*90)
    print(textwrap.fill(txt[:1500], width=100))

## 13. Optional extension: Anthropic evidence extraction agent

The baseline index is deterministic. This optional LLM step uses the Anthropic API only for **structured evidence extraction** from short district sections. The model returns a schema-constrained tool call with a tone label, confidence score, short quotes, and risks.

This keeps cost and token use low: the LLM is not asked to rebuild the index or reason over full reports. It only audits selected sections so students can compare model labels against the dictionary signal. Disagreement is not failure; it is a research signal.

In [ ]:
# Optional dependency for the Anthropic evidence extraction agent.
# Uncomment in a fresh environment:
# !pip install anthropic python-dotenv

import os
from pathlib import Path

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    # Lightweight fallback for classroom notebooks when python-dotenv is not installed.
    env_path = Path('.env')
    if env_path.exists():
        for line in env_path.read_text().splitlines():
            if line.strip() and not line.lstrip().startswith('#') and '=' in line:
                key, value = line.split('=', 1)
                os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

try:
    from anthropic import Anthropic
except ImportError:
    Anthropic = None

EVIDENCE_TOOL = {
    'name': 'extract_beige_book_evidence',
    'description': 'Extract an auditable tone label and supporting evidence from one Beige Book district section.',
    'input_schema': {
        'type': 'object',
        'properties': {
            'overall_tone': {
                'type': 'string',
                'enum': ['improving', 'stable', 'deteriorating', 'mixed'],
                'description': 'Tone of current economic conditions described in the section.'
            },
            'confidence': {
                'type': 'number',
                'minimum': 0,
                'maximum': 1,
                'description': 'Confidence in the tone label.'
            },
            'evidence': {
                'type': 'array',
                'items': {'type': 'string'},
                'minItems': 3,
                'maxItems': 3,
                'description': 'Three short direct quotes from the section supporting the label.'
            },
            'risks': {
                'type': 'array',
                'items': {'type': 'string'},
                'description': 'Major economic risks or concerns mentioned in the section.'
            }
        },
        'required': ['overall_tone', 'confidence', 'evidence', 'risks'],
        'additionalProperties': False,
    }
}


class AnthropicEvidenceAgent:
    def __init__(self, model='claude-3-5-haiku-latest', max_tokens=600):
        if Anthropic is None:
            raise ImportError('Install the Anthropic SDK first: pip install anthropic')
        if not os.getenv('ANTHROPIC_API_KEY'):
            raise ValueError('Set ANTHROPIC_API_KEY in .env or your shell before running the Anthropic evidence agent.')
        self.client = Anthropic()
        self.model = model
        self.max_tokens = max_tokens

    def extract_one(self, section_row: pd.Series) -> Dict:
        prompt = f"""
You are an economic research assistant. Given one Federal Reserve Beige Book district section, extract only what is supported by the text.

Release: {section_row['label']}
District: {section_row['district']}

District section:
{section_row['text']}

Use the extract_beige_book_evidence tool. Do not infer beyond the text. Evidence must be short direct quotes from the section.
""".strip()

        response = self.client.messages.create(
            model=self.model,
            max_tokens=self.max_tokens,
            temperature=0,
            tools=[EVIDENCE_TOOL],
            tool_choice={'type': 'tool', 'name': 'extract_beige_book_evidence'},
            messages=[{'role': 'user', 'content': prompt}],
        )

        tool_inputs = [block.input for block in response.content if block.type == 'tool_use']
        if not tool_inputs:
            raise ValueError('Anthropic response did not contain the expected tool call.')

        result = tool_inputs[0]
        return {
            'release_date': section_row['release_date'],
            'label': section_row['label'],
            'district': section_row['district'],
            **result,
        }

    def run(self, sections_to_extract: pd.DataFrame) -> pd.DataFrame:
        rows = [self.extract_one(row) for _, row in sections_to_extract.iterrows()]
        return pd.DataFrame(rows)

In [ ]:
# Run the LLM extractor on the latest release's weakest and strongest districts.
# This keeps the live API demo small, cheap, and easy to audit.
latest_label = release_index.iloc[-1]['label']
latest_ranked = indices['district_index'].query('label == @latest_label').sort_values('district_comparison_index')
selected_districts = [latest_ranked.iloc[0]['district'], latest_ranked.iloc[-1]['district']]

llm_sections = sections.query('label == @latest_label and district in @selected_districts').copy()

evidence_agent = AnthropicEvidenceAgent()
llm_evidence = evidence_agent.run(llm_sections)
llm_evidence

In [ ]:
# Compare the LLM tone labels with the deterministic dictionary signal.
comparison = llm_evidence.merge(
    latest_ranked[['district', 'district_comparison_index', 'raw_sentiment', 'uncertainty_rate']],
    on='district',
    how='left'
)
comparison[['district', 'overall_tone', 'confidence', 'district_comparison_index', 'raw_sentiment', 'uncertainty_rate', 'evidence', 'risks']]

## 14. Exercises

1. **Lexicon sensitivity:** remove 10 words from the positive and negative dictionaries. Does the ranking change?
2. **Normalization sensitivity:** compare the current district-adjusted index with a simple raw average.
3. **Topic analysis:** add a new topic for energy or trade policy.
4. **External validation:** compare the Beige Book index with payroll surprises, ISM, CPI surprises, or regional Fed survey data.
5. **Agent upgrade:** replace the dictionary scorer with a schema-constrained LLM scorer and compare cost, stability, and interpretability.
